In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
# Set up base paths
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis")
existing_future_forest_folder = base_path / "Processed_data/existing_future_forest"

In [ ]:
# Construct full file paths for your CSVs
existing_basin_stats_csv_path = existing_future_forest_folder / "existing_basin_stats.csv"
afforestable_basin_stats_csv_path = existing_future_forest_folder / "existing_and_non_agri_afforestable_basin_stats.csv"

# Read in the CSVs
existing_basin_stats = pd.read_csv(existing_basin_stats_csv_path)
afforestable_basin_stats = pd.read_csv(afforestable_basin_stats_csv_path)

In [ ]:
# Make sure both DataFrames have a common identifier (e.g., "HYBAS_ID").
# Then merge them on that identifier. The suffixes will differentiate the columns.
merged_stats = existing_basin_stats.merge(
    afforestable_basin_stats,
    on="HYBAS_ID",
    suffixes=('_existing', '_afforestable')
)

In [ ]:
# List of fields you want to compare (ensure these names match your DataFrame column names)
fields = [
    "Number of forest patches",      # patch_count
    "Average patch size (m2)",
    "Largest patch size (m2)",
    "Total edge length",
    "Average patch size (ha)",
    "Largest patch size (ha)",
    "Total catchment forest area (m2)",
    "Total catchment forest area (ha)",
    "Percentage forest in catchment"
]

In [ ]:
# First, convert the formatted strings back to numeric values for both DataFrames in merged_stats.
for field in fields:
    for suffix in ["_existing", "_afforestable"]:
        colname = field + suffix
        # Check if the column is of type object (string) before using .str.replace()
        if merged_stats[colname].dtype == "object":
            merged_stats[colname] = pd.to_numeric(
                merged_stats[colname].str.replace(",", ""), errors='coerce'
            )
        else:
            merged_stats[colname] = pd.to_numeric(merged_stats[colname], errors='coerce')

# Calculate differences for each field (afforestable minus existing)
for field in fields:
    diff_field = "diff_" + field.replace(" ", "_")  # e.g., diff_Average_patch_size_(m2)
    merged_stats[diff_field] = merged_stats[field + "_afforestable"] - merged_stats[field + "_existing"]

# Inspect the differences; for example, show HYBAS_ID and all the difference columns
diff_columns = ["HYBAS_ID"] + [ "diff_" + field.replace(" ", "_") for field in fields ]
display(merged_stats[diff_columns])

In [ ]:

# Build the output file path
output_file = existing_future_forest_folder / "merged_diff_stats.csv"

# Save the DataFrame as a CSV file without the index
merged_stats.to_csv(output_file, index=False)